# Nexus 2.0 Notebook Replica

Este notebook replica una arquitectura tipo Nexus 2.0 usando:
- Foundry / Azure OpenAI compatible endpoint
- Power BI REST API para ejecutar DAX
- Orquestador multiagente en Python

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

repo_root = Path.cwd().parent  # si estás dentro de notebooks/
load_dotenv(repo_root / ".env")

print(os.getenv("AZURE_AI_FOUNDRY_ENDPOINT"))
print(os.getenv("AZURE_AI_FOUNDRY_DEPLOYMENT"))

key = os.getenv("AZURE_AI_FOUNDRY_API_KEY")
print(key[:10] + "..." if key else "API KEY NOT FOUND")

https://aoai-pocai-eastus2-dev-01.openai.azure.com/
gpt-5-mini
1IyXRttuSk...


In [2]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent  # si estás en /nexus2.0/notebooks
sys.path.insert(0, str(repo_root))

print(repo_root)

c:\Users\SamuelAguilarRamirez\nexus2.0


In [3]:
import src.llm_client
print(src.llm_client.__file__)

c:\Users\SamuelAguilarRamirez\nexus2.0\src\llm_client.py


In [4]:
from src.llm_client import AzureAIFoundry

llm = AzureAIFoundry()

response = llm.chat(
    system_prompt="You are a test assistant.",
    user_prompt="Respond OK only."
)

print(response)

OK


In [5]:
from pathlib import Path

path = Path(r"C:\Users\SamuelAguilarRamirez\nexus2.0\src\agents\intent_clarifier.py")

text = path.read_text(encoding="utf-8")
text = text.replace(
    "from prompts.intent_clarifier import INTENT_SYSTEM_PROMPT",
    "from src.prompts.intent_clarifier import INTENT_SYSTEM_PROMPT"
)
path.write_text(text, encoding="utf-8")

print("Fixed import")

Fixed import


In [6]:
import importlib
import src.agents.intent_clarifier
importlib.reload(src.agents.intent_clarifier)

from src.agents.intent_clarifier import IntentClarifierAgent

In [7]:
intent = IntentClarifierAgent(
    llm,
    general_syn="NSR = Net Sales Revenue; market = Ship To",
    dav="Available measures: NSR, Volume. Available dimensions: Period, Ship To, Product, Channel."
).run("Show NSR YTD by channel for Colombia")

print(intent)

{'intent': 'semantic_query', 'agents': [{'name': 'FHB_dataset', 'instruction': 'Retrieve NSR (Net Sales Revenue - SELL-IN) Year-to-Date (YTD) by Channel for Ship To = Colombia. Use Actuals scenario (Metrics-Actuals-Rev). Time: compute YTD up to the latest Period available in the model. Group results by Channel and return a table with columns: Channel, NSR_YTD. Do not add any other filters or comparisons.'}, {'name': 'VisualizationAgent', 'instruction': 'Create a visualization of the returned NSR YTD by Channel for Colombia. The visualization should be suitable for comparing values across channels (suggest a bar chart as the default). Include channel on the x-axis and NSR_YTD on the y-axis, display numeric values on bars, and format NSR as currency. Provide JSON instructions for rendering the chart. If the user prefers a different chart type, allow switching.'}], 'needs_visualization': True, 'output_format': 'chart', 'business_question': 'NSR (Net Sales Revenue - SELL-IN) Year-to-Date b

In [8]:
fhb_instruction = intent["agents"][0]["instruction"]
print(fhb_instruction)

Retrieve NSR (Net Sales Revenue - SELL-IN) Year-to-Date (YTD) by Channel for Ship To = Colombia. Use Actuals scenario (Metrics-Actuals-Rev). Time: compute YTD up to the latest Period available in the model. Group results by Channel and return a table with columns: Channel, NSR_YTD. Do not add any other filters or comparisons.


In [9]:
from src.agents.dax_query_developer import DaxQueryDeveloperAgent

developer = DaxQueryDeveloperAgent(
    llm,
    general_syn="NSR = Net Sales Revenue; market = Ship To",
    dav = """
Tables and columns available:

Period:
- Period[Date]
- Period[Year]
- Period[Month]
- Period[Week]

Ship To:
- Ship To[Country]

Channel:
- Channel[Channel]

Measures:
- [NSR]
- [NSR YTD] (predefined measure)
Available measures: NSR, Volume. Available dimensions: Period, Ship To, Product, Channel."""
)

dax_plan = developer.run(fhb_instruction)

print(dax_plan)

EVALUATE
VAR LatestDate = CALCULATE(MAX(Period[Date]), ALL(Period))
RETURN
SUMMARIZECOLUMNS(
    Channel[Channel],
    Ship To[Country] = "Colombia",
    Period[Date] <= LatestDate,
    "NSR_YTD", [NSR YTD]
)


In [10]:
from src.agents.dax_validator import DaxValidatorAgent

semantic_context = """
Target model: NSR LATAM semantic model

Tables and columns available:

Period:
- Period[Date]
- Period[Year]
- Period[Month]
- Period[Week]

Ship To:
- Ship To[Country]

Channel:
- Channel[Channel]

Measures:
- [NSR]
- [NSR YTD]

Rules:
- NSR is SELL-IN
- Use Period[Date] for time intelligence
"""

validator = DaxValidatorAgent(
    llm_client=llm,
    semantic_context=semantic_context
)

validation_result = validator.run(
    business_question="Get NSR YTD by channel for Colombia",
    dax_query=dax_plan 
)

print(validation_result)

NOT APPROVED

Required fixes:
1. Quote the table name that contains a space. Change Ship To[Country] = "Colombia" to 'Ship To'[Country] = "Colombia".

Instructions for DAX Developer:
- Fix ONLY the issue listed above.
- Do NOT change any other part of the query.


In [11]:
from pathlib import Path

root = Path(r"C:\Users\SamuelAguilarRamirez\nexus2.0")

matches = list(root.rglob("Microsoft.AnalysisServices.AdomdClient.dll"))

for m in matches:
    print(m)

C:\Users\SamuelAguilarRamirez\nexus2.0\lib\Microsoft.AnalysisServices.AdomdClient.dll


In [ ]:
from dotenv import load_dotenv
load_dotenv()

from src.connections.nsr_conn import AdomdConnector, build_connection_string, Config
from src.agents.dax_executor import DaxExecutorAgent

# Crear conexión
nsr_conn = AdomdConnector(
    dll_path=Config.DLL_PATH,
    connection_string=build_connection_string()
)

# Crear agente
executor = DaxExecutorAgent(nsr_conn)

# Test query
query = """
EVALUATE
ROW("Test", 1)
"""

df = executor.run(query)
display(df.head())

✅ Entorno .NET y DLL preparados.
❌ Error en la consulta: When interactive authentication is not supported, an external access-token is required; either provide it in the connection-string or by setting the AccessToken property.
   at Microsoft.AnalysisServices.AdomdClient.ConnectionInfo.AcquireAADTokenAndResolveHTTPConnectionPropertiesForPaaSInfrastructure(IConnectivityOwner owner, Uri& dataSourceUri)
   at Microsoft.AnalysisServices.AdomdClient.HttpStream.Create(IConnectivityOwner owner, ConnectionInfo info, XmlaDataType desiredRequestType, XmlaDataType desiredResponseType, Int32 readTimeout, Boolean& isSessionTokenNeeded)
   at Microsoft.AnalysisServices.AdomdClient.XmlaClient.OpenHttpConnection(ConnectionInfo connectionInfo, Boolean& isSessionTokenNeeded)
   at Microsoft.AnalysisServices.AdomdClient.XmlaClient.OpenConnectionAndCheckIfSessionTokenNeeded(ConnectionInfo connectionInfo)
   at Microsoft.AnalysisServices.AdomdClient.XmlaClient.OpenConnection(ConnectionInfo connectionInfo,

RuntimeError: DAX execution failed. Check ADOMD connection or query syntax.